# SymboLM vs. DeepSeek-R1 Base: Official Apples-to-Apples Benchmark on GPU
**Author & Principal Systems Architect:** Sanish Gyawali (Rupandehi, Nepal)  
**AI Systems Collaborator:** Antigravity (Google DeepMind)  
**Repository:** [https://github.com/gyawalisanish0/symboLM](https://github.com/gyawalisanish0/symboLM)  

### Objectives:
Direct 3-arm empirical comparison between:
1. **Arm 1 (Base R1 Unconstrained @ 4096 tokens):** Replicates DeepSeek's published baseline accuracy; quantifies natural language token inflation.
2. **Arm 2 (Base R1 Budget-Choked @ 512 tokens):** Proves edge/low-latency collapse and premature truncation failure of conversational CoT.
3. **Arm 3 (SymboLM v2.1 Grammar FSM @ 512 tokens):** Demonstrates high-density symbolic reasoning, zero truncation, and 75%+ token compression.

**Evaluated on Gold-Standard Benchmarks:**
- `HuggingFaceH4/MATH-500` (500 competition math problems across 7 domains)
- `AI-MO/aimo-validation-aime` (AIME 2024 Olympiad, 30 problems)

In [ ]:
# Cell 1: Environment & GPU Verification
import os
import sys
import torch

print('Python version:', sys.version)
print('Torch version :', torch.__version__)
if torch.cuda.is_available():
    print(f'[OK] NVIDIA CUDA detected! Found {torch.cuda.device_count()} GPU(s):')
    for i in range(torch.cuda.device_count()):
        mem = torch.cuda.get_device_properties(i).total_memory / 1e9
        print(f'  GPU {i}: {torch.cuda.get_device_name(i)} ({mem:.1f} GB VRAM)')
else:
    print('[Warning] No CUDA GPU found!')


In [ ]:
# Cell 2: Dependencies Installation
!pip install -q --upgrade pip
!pip uninstall -y -q torchao
!pip install -q 'transformers>=4.48.0' 'peft>=0.14.0' 'accelerate>=1.2.0' 'sympy>=1.12' 'datasets>=3.2.0'


In [ ]:
# Cell 3: Clone Latest SymboLM Repository from GitHub
!rm -rf /kaggle/working/symboLM
!git clone https://github.com/gyawalisanish0/symboLM.git /kaggle/working/symboLM
%cd /kaggle/working/symboLM
!git log -n 1 --oneline


In [ ]:
# Cell 5: Execute 3-Arm Apples-to-Apples Benchmark on Stratified 100 MATH-500
!python /kaggle/working/symboLM/eval/benchmark_apples_to_apples.py \
  --sample_limit 100 \
import shutil
from pathlib import Path

target_dir = Path('/kaggle/working/symboLM/checkpoints/symbolm_v2_adapter')
target_dir.mkdir(parents=True, exist_ok=True)

found = False
# Search for uncompressed adapter
for p in Path('/kaggle/input').glob('**/symbolm_v2_adapter'):
    if (p / 'adapter_config.json').exists() and (p / 'adapter_model.safetensors').exists():
        print(f'[OK] Found uncompressed v2 adapter at: {p}')
        for item in p.iterdir():
            if item.is_file():
                shutil.copy2(item, target_dir / item.name)
        found = True
        break

# Search for tarball archive
if not found:
    for arc in Path('/kaggle/input').glob('**/*v2*contrastive*policy*.tar.gz'):
        print(f'[OK] Found v2 policy tar archive at: {arc}. Unpacking...')
        os.system(f'tar -xzf "{arc}" -C "{target_dir}"')
        found = True
        break

print('[Status] Validating SymboLM v2 Adapter Files:')
for f in sorted(target_dir.iterdir()):
    sz = f.stat().st_size / 1e6 if f.is_file() else 0
    print(f'  {f.name} ({sz:.2f} MB)' if f.is_file() else f'  {f.name}/')

assert (target_dir / 'adapter_config.json').exists(), 'CRITICAL: adapter_config.json missing!'
assert (target_dir / 'adapter_model.safetensors').exists(), 'CRITICAL: adapter_model.safetensors missing!'
print('[VERIFIED] SymboLM Policy Adapter successfully verified!')


In [ ]:
# Cell 5: Execute 3-Arm Apples-to-Apples Benchmark on Official MATH-500
!python /kaggle/working/symboLM/eval/benchmark_apples_to_apples.py \
  --dataset math500 \
  --base_model deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B \
  --adapter_path /kaggle/working/symboLM/checkpoints/symbolm_v2_adapter \
  --output_json /kaggle/working/math500_apples_to_apples_results.json \
  --output_report /kaggle/working/MATH500_APPLES_TO_APPLES_REPORT.md


In [ ]:
# Cell 6: Execute 3-Arm Apples-to-Apples Benchmark on Official AIME 2024 Olympiad
!CUDA_VISIBLE_DEVICES=0 python /kaggle/working/symboLM/eval/benchmark_apples_to_apples.py \
  --dataset aime2024 \
  --base_model deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B \
  --adapter_path /kaggle/working/symboLM/checkpoints/symbolm_v2_adapter \
  --output_json /kaggle/working/aime2024_apples_to_apples_results.json \
  --output_report /kaggle/working/AIME2024_APPLES_TO_APPLES_REPORT.md


In [ ]:
# Cell 7: Archive All Benchmark Receipts & Reports
!tar -czf /kaggle/working/symbolm_apples_to_apples_receipts.tar.gz -C /kaggle/working \
  math500_apples_to_apples_results.json \
  MATH500_APPLES_TO_APPLES_REPORT.md \
  aime2024_apples_to_apples_results.json \
  AIME2024_APPLES_TO_APPLES_REPORT.md

print('[COMPLETE] All Apples-to-Apples benchmark receipts archived to /kaggle/working/symbolm_apples_to_apples_receipts.tar.gz')
